In [2]:
import imageio
import numpy as np

def face_segment(segment_part, mask_path):
    face_segment_anno = imageio.v2.imread(mask_path)

    face_segment_anno = np.array(face_segment_anno)
    bg = (face_segment_anno == 0)
    skin = (face_segment_anno == 1)
    l_brow = (face_segment_anno == 2)
    r_brow = (face_segment_anno == 3)
    l_eye = (face_segment_anno == 4)
    r_eye = (face_segment_anno == 5)
    eye_g = (face_segment_anno == 6)
    l_ear = (face_segment_anno == 7)
    r_ear = (face_segment_anno == 8)
    ear_r = (face_segment_anno == 9)
    nose = (face_segment_anno == 10)
    mouth = (face_segment_anno == 11)
    u_lip = (face_segment_anno == 12)
    l_lip = (face_segment_anno == 13)
    neck = (face_segment_anno == 14)
    neck_l = (face_segment_anno == 15)
    cloth = (face_segment_anno == 16)
    hair = (face_segment_anno == 17)
    hat = (face_segment_anno == 18)
    face = np.logical_or.reduce((skin, l_brow, r_brow, l_eye, r_eye, eye_g, l_ear, r_ear, ear_r, nose, mouth, u_lip, l_lip))

    if segment_part == 'faceseg_bg':
        seg_m = bg
    elif segment_part == 'faceseg_fg':
        seg_m = ~bg
    else: raise NotImplementedError(f"Segment part: {segment_part} is not found!")
    
    out = seg_m
    return out
            
def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

def blending_mask(img_path, mask_path, hdr_from_bg_path):
    if isinstance(mask_path, np.ndarray):
        mask = mask_path / 255.0
    else:
        mask = imageio.v2.imread(mask_path) / 255.0 #[256, 256]
        
    if isinstance(img_path, np.ndarray):
        img = img_path / 255.0
    else:
        img = imageio.v2.imread(img_path).astype(np.float32) / 255.0
        
    blurred_mask = cv2.GaussianBlur(mask, (3, 3), 0)
    # Apply erosion
    kernel = np.ones((3,3),np.uint8)
    eroded_mask = cv2.erode(blurred_mask, kernel, iterations = 1)
    mask = eroded_mask [..., np.newaxis]
    
    bg = imageio.v2.imread(hdr_from_bg_path).astype(np.float32) / 255.0
    # alpha blending
    out = img * mask + bg * (1 - mask)
    out = (out * 255).astype(np.uint8)
    return out

In [40]:
import numpy as np
import torch as th
import json, os, glob
import imageio
import cv2
import multiprocessing as mp
import matplotlib.pyplot as plt
from PIL import Image

# Azimuth = Axis 1
method = ["neural_gaffer_azimuth", "difareli++_axis1_color_SD75_0.8C_Lmax10"]
meta = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/MajorRevision/hdr/hdr_finale_axis=1.json"))
sample = json.load(open("/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/all_rotateSH.json"))
out_dir = "./rotate_hdr_axis=1_figure/"
os.makedirs(out_dir, exist_ok=True)
# default_fidx = [33, 26, 17, 6]  # 0-59
default_fidx = [1, 10, 20, 30]  # 0-59

# hdr_map = ["012_hdrmaps_com_free_2K", "064_hdrmaps_com_free_2K", "117_hdrmaps_com_free_2K", "125_hdrmaps_com_free_2K", "128_hdrmaps_com_free_2K"]
hdr_map = ["064_hdrmaps_com_free_2K", "125_hdrmaps_com_free_2K"]

for pid, dat in sample['pair'].items():
    src = dat['src']
    dst = dat['dst']
    
    for hdr in hdr_map:
        hdr_fidx = dat.get(f'hdr={hdr}', default_fidx)
        out_combined = []
        fail = False
        for m in method:
            if m == "neural_gaffer_azimuth":
                path = meta[m]['res_dir']
                frame_path = f"{path}/{src.split('.')[0]}/{hdr}/"
                frames = sorted(glob.glob(f"{frame_path}/pred_*.png"))
                if len(frames) < 1:
                    print(f"Skip {pid}-{src.split('.')[0]} for {m} since no result is found.")
                    fail = True
                    break
                frames = [f'{frame_path}/pred_{i-1:04d}.png' for i in hdr_fidx]
                target_hdr = sorted(glob.glob(f"{path}/{src.split('.')[0]}/{hdr}/target_*.png"))
                mask = f"/data/mint/DPM_Dataset/Dataset_For_Baseline/NeuralGaffer/input_subject_finale/preprocessed/mask/{src.split('.')[0]}.png"
            elif m == "difareli++_axis1_color_SD75_0.8C_Lmax10":
                path = meta[m]['res_dir']
                frame_path = f"{path}/{hdr}/src={src}/dst={dst}/Lerp_1000/n_frames=60/"
                frames = sort_by_frame(glob.glob(f"{frame_path}/res_frame*.png"))[1:]
                if len(frames) < 1:
                    print(f"Skip {pid}-{src.split('.')[0]} for {m} since no result is found.")
                    fail = True
                    break
                frames = [f'{frame_path}/res_frame{i}.png' for i in hdr_fidx]
                render_frames = [f'{frame_path}/dst_ren_frame{i}.png' for i in hdr_fidx]
                shadow_frames = [f'{frame_path}/dst_shadm_shad_frame{i}.png' for i in hdr_fidx]
                assert len(shadow_frames) == len(render_frames)
                cond = [np.concatenate([imageio.v2.imread(render_frames[i]), imageio.v2.imread(shadow_frames[i])], axis=1) for i in range(len(render_frames))]
                cond = np.concatenate(cond, axis=1)
                
                target_hdr = sorted(glob.glob(f"{meta['neural_gaffer_azimuth']['res_dir']}/{src.split('.')[0]}/{hdr}/target_*.png"))
                mask = f"/data/mint/DPM_Dataset/ffhq_256_with_anno/face_segment_with_pupil/valid/anno/anno_{src.split('.')[0]}.png"
                mask = face_segment('faceseg_fg', mask) * 1.0
                
            # elif m == "difareli++_azimuth_grey":
            #     path = meta[m]['res_dir']
            #     frames = sort_by_frame(glob.glob(f"{path}/{hdr}/src={src}/dst={dst}/Lerp_1000/n_frames=60/res_frame*.png"))[1:][::-1]
            #     # print(frames)
            #     target_hdr = sorted(glob.glob(f"{meta['neural_gaffer_azimuth']['res_dir']}/{src.split('.')[0]}/{hdr}/target_*.png"))
            #     mask = f"/data/mint/DPM_Dataset/ffhq_256_with_anno/face_segment_with_pupil/valid/anno/anno_{src.split('.')[0]}.png"
            #     mask = face_segment('faceseg_fg', mask) * 1.0
                # print(mask.shape, np.min(mask), np.max(mask))
                
            target_hdr = [target_hdr[59-i] for i in hdr_fidx]
            
            # Reuse the target hdr as background for difareli++ too, but need to rescale first
            if "difareli++" in m:
                frames_tmp = [imageio.v2.imread(f) for f in frames] # 0 - 255
                mask_tmp = mask
                out_frames = []
                out_mask = []
                for i in range(len(frames_tmp)):
                    proc_img = frames_tmp[i]
                    proc_img = np.concatenate([proc_img, mask_tmp[..., np.newaxis] * 255], axis=-1)
                    
                    x, y, w, h = cv2.boundingRect((mask_tmp*255).astype(np.uint8))
                    max_size = max(w, h)
                    ratio = 0.75
                    side_len = int(max_size / ratio)
                    padded_image = np.zeros((side_len, side_len, 4), dtype=np.uint8)
                    center = side_len//2
                    padded_image[center-h//2:center-h//2+h, center-w//2:center-w//2+w] = proc_img[y:y+h, x:x+w]
                    rgba = np.array(Image.fromarray(padded_image).resize((256, 256), Image.LANCZOS))
                    # rgba is 0 - 255 for each channel
                    rgba_arr = np.array(rgba) / 255.0   # normalized to 0 - 1
                    rgb = rgba_arr[...,:3] * rgba_arr[...,-1:] + (1 - rgba_arr[...,-1:])    # 256, 256, 3; 0-1
                    mask = rgba_arr[...,-1:]    # 256, 256, 1; 0-1
                    out_frames.append((rgb * 255).astype(np.uint8))
                    out_mask.append((mask[...,0] * 255).astype(np.uint8))
                frames = out_frames
                mask = out_mask[0]
            
            with mp.Pool(5) as p:
                out_frames = p.starmap(blending_mask, (zip(frames, [mask]*5, target_hdr)))
            out_frames = np.concatenate(out_frames, axis=1)
            out_combined.append(out_frames)
            
        if not fail:
            
            out_combined = np.concatenate(out_combined, axis=0)
            # Resize cond's widgth to match out_combined and append to the bottom
            cond = cv2.resize(cond, (out_combined.shape[1], int(cond.shape[0] * out_combined.shape[1] / cond.shape[1])), interpolation=cv2.INTER_LANCZOS4)
            out_combined = np.concatenate([out_combined, cond], axis=0)
            os.makedirs(f"{out_dir}/{hdr}/", exist_ok=True)
            Image.fromarray(out_combined).save(f"{out_dir}/{hdr}/{pid}_{src.split('.')[0]}_{hdr}_axis1.png")
            

Skip pair5-66943 for neural_gaffer_azimuth since no result is found.
Skip pair5-66943 for neural_gaffer_azimuth since no result is found.
Skip pair6-63295 for neural_gaffer_azimuth since no result is found.
Skip pair6-63295 for neural_gaffer_azimuth since no result is found.
Skip pair7-68018 for neural_gaffer_azimuth since no result is found.
Skip pair7-68018 for neural_gaffer_azimuth since no result is found.
Skip pair8-62028 for neural_gaffer_azimuth since no result is found.
Skip pair8-62028 for neural_gaffer_azimuth since no result is found.
Skip pair9-64240 for neural_gaffer_azimuth since no result is found.
Skip pair9-64240 for neural_gaffer_azimuth since no result is found.
Skip pair10-65112 for neural_gaffer_azimuth since no result is found.
Skip pair10-65112 for neural_gaffer_azimuth since no result is found.
Skip pair11-64334 for neural_gaffer_azimuth since no result is found.
Skip pair11-64334 for neural_gaffer_azimuth since no result is found.
Skip pair12-64036 for neural_g